In [12]:
import torch
from transformers import AutoTokenizer
from vul_detector import VulDetector

MAX_LEN = 512
# -------------------------
# 6) Inference function
# -------------------------
def predict_vulnerability(
    code_text: str,
    model_path: str = None,
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    return_probabilities: bool = False,
    threshold: float = 0.5,          # ✅ add threshold
    use_argmax: bool = False,        # ✅ allow argmax if you want
    vulnerable_label: int = 1        # ✅ make mapping explicit
):
    """Run a single vulnerability prediction."""
    model_to_use = model_instance if model_instance is not None else model
    tokenizer_to_use = tokenizer_instance if tokenizer_instance is not None else tokenizer
    device_to_use = device_instance if device_instance is not None else device

    if model_path is not None:
        checkpoint = torch.load(model_path, map_location=device_to_use)
        model_to_use.load_state_dict(checkpoint)
        model_to_use.to(device_to_use)
        model_to_use.eval()

    inputs = tokenizer_to_use(
        code_text.strip(),
        truncation=True,
        max_length=MAX_LEN,
        padding=True,                 # ✅ dynamic padding for single input
        add_special_tokens=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device_to_use) for k, v in inputs.items()}

    model_to_use.eval()
    with torch.no_grad():
        outputs = model_to_use(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

        vuln_prob = probs[0, vulnerable_label].item()

        if use_argmax:
            predicted_class = torch.argmax(probs, dim=-1).item()
        else:
            predicted_class = vulnerable_label if vuln_prob >= threshold else 1 - vulnerable_label

    if return_probabilities:
        return {
            "safe": probs[0][0].item(),
            "vulnerable": probs[0][1].item(),
            "vuln_prob": vuln_prob,
            "threshold": threshold,
            "predicted_class": predicted_class
        }

    return "vulnerable" if predicted_class == vulnerable_label else "safe"


In [23]:
from pathlib import Path

model_name = "microsoft/unixcoder-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = VulDetector(model_name=model_name, num_labels=2)

checkpoint_path = Path("model/best_model_epoch_5.pt")
print(checkpoint_path.is_file())
if checkpoint_path.is_file():
    state_dict = torch.load(checkpoint_path, map_location=device)
    #model.load_state_dict(state_dict)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    w = model.model.classifier.out_proj.weight
    b = model.model.classifier.out_proj.bias
    print("head weight mean(abs):", w.abs().mean().item())
    print("head bias mean(abs):", b.abs().mean().item())


else:
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

model.to(device)
model.eval()

# sample_code = """void f(char *src) { char buf[8]; strcpy(buf, src); }"""
# # sample_code = """
# #     void fz_init_cached_color_converter(fz_context *ctx, fz_color_converter *cc, fz_colorspace *is, fz_colorspace *ds, fz_colorspace *ss, const fz_color_params *params)\n{\n\tint n = ss->n;\n\tfz_cached_color_converter *cached = fz_malloc_struct(ctx, fz_cached_color_converter);\n\n\tcc->opaque = cached;\n\tcc->convert = fz_cached_color_convert;\n\tcc->ds = ds ? ds : fz_device_gray(ctx);\n\tcc->ss = ss;\n\tcc->is = is;\n\n\tfz_try(ctx)\n\t{\n\t\tfz_find_color_converter(ctx, &cached->base, is, cc->ds, ss, params);\n\t\tcached->hash = fz_new_hash_table(ctx, 256, n * sizeof(float), -1, fz_free);\n\t}\n\tfz_catch(ctx)\n\t{\n                fz_drop_color_converter(ctx, &cached->base);\n                fz_drop_hash_table(ctx, cached->hash);\n                fz_free(ctx, cached);\n                fz_rethrow(ctx);\n        }\n }
# # """
# code_text = "Analyze this C function for vulnerability:\n" + sample_code

# result = predict_vulnerability(
#     sample_code,
#     model_instance=model.model,
#     tokenizer_instance=tokenizer,
#     device_instance=device,
#     return_probabilities=True,
#     threshold=0.23
# )
# print("\nSingle prediction:", result)
sample_code = "void f(char *src) { char buf[8]; strcpy(buf, src); }"
code_text = "Analyze this C function for vulnerability:\n" + sample_code

r1 = predict_vulnerability(sample_code, model_instance=model, tokenizer_instance=tokenizer,
                           device_instance=device, return_probabilities=True, threshold=0.23)
r2 = predict_vulnerability(code_text, model_instance=model, tokenizer_instance=tokenizer,
                           device_instance=device, return_probabilities=True, threshold=0.23)

print("RAW:", r1)
print("PROMPT:", r2)





1026


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


True
Missing keys: []
Unexpected keys: []
head weight mean(abs): 0.015805156901478767
head bias mean(abs): 0.000726129102986306
RAW: {'safe': 0.995087206363678, 'vulnerable': 0.004912742879241705, 'vuln_prob': 0.004912742879241705, 'threshold': 0.23, 'predicted_class': 0}
PROMPT: {'safe': 0.9963829517364502, 'vulnerable': 0.0036170303355902433, 'vuln_prob': 0.0036170303355902433, 'threshold': 0.23, 'predicted_class': 0}
